In [6]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from tensorflow import keras

artifact_dir = Path('artifacts')
preprocessor = joblib.load(artifact_dir / 'hybrid_preprocessor.joblib')
label_encoder = joblib.load(artifact_dir / 'label_encoder.joblib')
encoder = keras.models.load_model(artifact_dir / 'hybrid_encoder.keras', compile=False)
model = keras.models.load_model(artifact_dir / 'hybrid_model.keras', compile=False)

feature_columns = [
    'Age',
    'Gender',
    'Hours_Studied',
    'Attendance',
    'Sleep_Hours',
    'Stress_Level',
    'Screen_Time',
    'Previous_GPA',
    'Part_Time_Job',
    'Study_Method',
    'Diet_Quality',
    'Internet_Quality',
    'Extracurricular',
    'Tutoring_Sessions_Per_Week',
    'Family_Income_Level',
    'Exam_Anxiety_Score',
]

def predict_grade(new_student: dict):
    row = dict(new_student)
    row.pop('Student_ID', None)
    input_df = pd.DataFrame([row])
    input_df = input_df.reindex(columns=feature_columns)
    input_df = input_df.fillna({
        'Age': 0,
        'Gender': 'Unknown',
        'Hours_Studied': 0.0,
        'Attendance': 0.0,
        'Sleep_Hours': 0.0,
        'Stress_Level': 0.0,
        'Screen_Time': 0.0,
        'Previous_GPA': 0.0,
        'Part_Time_Job': 'No',
        'Study_Method': 'Online',
        'Diet_Quality': 'Average',
        'Internet_Quality': 'Average',
        'Extracurricular': 'No',
        'Tutoring_Sessions_Per_Week': 0,
        'Family_Income_Level': 'Middle',
        'Exam_Anxiety_Score': 0.0,
    })

    x_fw = preprocessor.transform(input_df)
    z_new = encoder.predict(x_fw, verbose=0)
    z_new_seq = z_new.reshape(z_new.shape[0], z_new.shape[1], 1)
    y_new_proba = model.predict(z_new_seq, verbose=0)
    y_new_pred = np.argmax(y_new_proba, axis=1)
    predicted_grade = label_encoder.inverse_transform(y_new_pred)[0]

    return predicted_grade, y_new_proba[0], input_df

In [ ]:
feature_specs = [
    {'key': 'Age', 'label': 'Age', 'type': 'int', 'default': 21},
    {'key': 'Gender', 'label': 'Gender', 'type': 'choice', 'default': 'Female', 'options': ['Female', 'Male', 'Non-Binary']},
    {'key': 'Hours_Studied', 'label': 'Hours Studied', 'type': 'float', 'default': 6.0},
    {'key': 'Attendance', 'label': 'Attendance', 'type': 'float', 'default': 85.0},
    {'key': 'Sleep_Hours', 'label': 'Sleep Hours', 'type': 'float', 'default': 7.0},
    {'key': 'Stress_Level', 'label': 'Stress Level', 'type': 'float', 'default': 3.0},
    {'key': 'Screen_Time', 'label': 'Screen Time', 'type': 'float', 'default': 3.5},
    {'key': 'Previous_GPA', 'label': 'Previous GPA', 'type': 'float', 'default': 3.0},
    {'key': 'Part_Time_Job', 'label': 'Part Time Job', 'type': 'choice', 'default': 'No', 'options': ['No', 'Yes']},
    {'key': 'Study_Method', 'label': 'Study Method', 'type': 'choice', 'default': 'Online', 'options': ['Online', 'Offline', 'Hybrid']},
    {'key': 'Diet_Quality', 'label': 'Diet Quality', 'type': 'choice', 'default': 'Average', 'options': ['Poor', 'Average', 'Good']},
    {'key': 'Internet_Quality', 'label': 'Internet Quality', 'type': 'choice', 'default': 'Good', 'options': ['Poor', 'Average', 'Good', 'Excellent']},
    {'key': 'Extracurricular', 'label': 'Extracurricular', 'type': 'choice', 'default': 'Yes', 'options': ['No', 'Yes']},
    {'key': 'Tutoring_Sessions_Per_Week', 'label': 'Tutoring Sessions Per Week', 'type': 'int', 'default': 1},
    {'key': 'Family_Income_Level', 'label': 'Family Income Level', 'type': 'choice', 'default': 'Middle', 'options': ['Low', 'Middle', 'High']},
    {'key': 'Exam_Anxiety_Score', 'label': 'Exam Anxiety Score', 'type': 'float', 'default': 2.0},
]


def ask_value(spec):
    prompt = f"{spec['label']}"
    if spec.get('options'):
        prompt += f" {spec['options']}"
    prompt += f" [{spec['default']}]: "

    while True:
        raw = input(prompt).strip()
        if raw == '':
            return spec['default']

        try:
            if spec['type'] == 'int':
                return int(raw)
            if spec['type'] == 'float':
                return float(raw)
            if spec['type'] == 'choice':
                if raw not in spec['options']:
                    print(f"Invalid value. Choose one of: {spec['options']}")
                    continue
                return raw
            return raw
        except ValueError:
            print(f"Invalid value for {spec['label']}. Try again.")


new_student = {'Student_ID': 'STU_NEW'}
for spec in feature_specs:
    new_student[spec['key']] = ask_value(spec)

predicted_grade, probabilities, input_df = predict_grade(new_student)

print('Prediction input:')
display(input_df)
print('Prediction:', predicted_grade)
print('Probabilities by class:')
print(pd.Series(probabilities, index=label_encoder.classes_).sort_values(ascending=False))


Prediction input:


,Age,Gender,Hours_Studied,Attendance,Sleep_Hours,Stress_Level,Screen_Time,Previous_GPA,Part_Time_Job,Study_Method,Diet_Quality,Internet_Quality,Extracurricular,Tutoring_Sessions_Per_Week,Family_Income_Level,Exam_Anxiety_Score
0,21,Male,1.0,0.0,1.0,1.0,7.0,1.0,Yes,Offline,Poor,Poor,No,0,Low,8.0


Prediction: Fail
Probabilities by class:
Fail    0.906402
D       0.062724
C       0.020958
B       0.006008
A       0.003908
dtype: float32
